<a href="https://colab.research.google.com/github/syedmahmoodiagents/genai_classes/blob/main/AgentBasics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_huggingface --q

In [25]:
from langchain_core.messages import HumanMessage, ToolMessage

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool

In [7]:
import os

In [8]:
os.environ["HF_TOKEN"] = "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123"

# Tool with docstrings

In [3]:
def calculate_multiply(a: int, b: int) -> int:
    """Multiplies two given integers together and returns the product."""
    return a * b

In [9]:
model = ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [10]:
# Bind the tool directly to the model instance
model_with_tools = model.bind_tools([calculate_multiply])

In [11]:
ai_msg = model_with_tools.invoke([HumanMessage(content="What is 35 multiplied by 12?")])

In [12]:
print(ai_msg.tool_calls)

[{'name': 'calculate_multiply', 'args': {'a': 35, 'b': 12}, 'id': 'fc_878e1a23-db2d-466c-8fe8-6a4820945080', 'type': 'tool_call'}]


In [28]:
ai_msg.content

''

In [19]:
tool_calls = ai_msg.tool_calls
tool_ca = tool_calls[0]

In [20]:
tool_calls

[{'name': 'calculate_multiply',
  'args': {'a': 35, 'b': 12},
  'id': 'fc_878e1a23-db2d-466c-8fe8-6a4820945080',
  'type': 'tool_call'}]

In [21]:
args = tool_ca["args"]
call_id = tool_ca["id"]

In [22]:
result = calculate_multiply(a=args["a"], b=args["b"])
print(3, result)

3 420


In [23]:
call_id

'fc_878e1a23-db2d-466c-8fe8-6a4820945080'

In [26]:
tool_message = ToolMessage(content=str(result), tool_call_id=call_id)
print(4, tool_message)

4 content='420' tool_call_id='fc_878e1a23-db2d-466c-8fe8-6a4820945080'


In [27]:
final_response = model_with_tools.invoke([HumanMessage(content="What is 35 multiplied by 12?"), ai_msg, tool_message])

print(5, final_response.content)

5 35 multiplied by 12 equals **420**.


# One more example this time using @tool

In [29]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))


@tool
def get_weather(city: str) -> str:
    """Return weather information."""
    return f"The weather in {city} is 28°C and Sunny."

In [30]:
TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather
}

In [40]:
alltools = [calculator, get_weather]

In [41]:
llm = ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [42]:
agent = model.bind_tools(alltools)

In [43]:
response = agent.invoke("What is the weather in Tokyo?")

In [44]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_weather', 'description': None}, 'id': 'fc_f50cd7d3-3cdd-4f3b-b101-2b6250f50278', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 143, 'total_tokens': 197}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_f102a3179c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ffeae-3a23-7c02-b55d-d7202edb6973-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'fc_f50cd7d3-3cdd-4f3b-b101-2b6250f50278', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 54, 'total_tokens': 197})

In [35]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Tokyo'},
  'id': 'fc_dacdcc91-cae7-484e-8dd2-9d9439973e5e',
  'type': 'tool_call'}]

In [36]:
response.tool_calls[0]

{'name': 'get_weather',
 'args': {'city': 'Tokyo'},
 'id': 'fc_dacdcc91-cae7-484e-8dd2-9d9439973e5e',
 'type': 'tool_call'}

In [37]:
tool_ca = response.tool_calls[0]

In [ ]:
tool_result = get_weather.invoke(tool_ca["args"])

In [38]:
tool_result

'The weather in Tokyo is 28°C and Sunny.'

In [39]:
final_response = agent.invoke([
    {"role": "user", "content": "What is the weather in Tokyo?"},
    response,
    {"role": "tool", "content": str(tool_result), "tool_call_id": tool_ca["id"]}
])

print(final_response)

content='The weather in Tokyo is 28\u202f°C and sunny.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 179, 'total_tokens': 233}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ffea8-0d3d-7f10-8435-04b4c2363a16-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 179, 'output_tokens': 54, 'total_tokens': 233}


# Lets try to simplify this....

In [45]:
from langchain.agents import create_agent

In [54]:
llm2 = ChatHuggingFace(llm = HuggingFaceEndpoint(repo_id= "openai/gpt-oss-20b"))

In [55]:
@tool
def add(a: int, b: int)->int:
    """add function takes two integers and returns the sum of them"""
    return a+b

In [56]:
tools = [add]

In [58]:
agent = create_agent(llm2, tools,
    system_prompt="Please use tool of add function to calculate" # this is a sys prompt
)

In [63]:
response = agent.invoke({"input": "What is the addition of 3 and 7?"})

In [60]:
response['messages']

[AIMessage(content="Sure! Could you let me know which two numbers you'd like to add together? Once I have them, I'll use the `add` function to compute the result.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 136, 'total_tokens': 270}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffecc-d290-7ce3-b721-3caf54157af8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 134, 'total_tokens': 270})]

In [61]:
# print(response['messages'])
for m in response['messages']:
    print(m.content)
    print("_________")

Sure! Could you let me know which two numbers you'd like to add together? Once I have them, I'll use the `add` function to compute the result.
_________
